# 🗞️ Türkçe Haber Sınıflandırma — DistilBERT Fine-Tuning

**Veri Seti:** `batubayk/TR-News`  
**Model:** `dbmdz/distilbert-base-turkish-cased`  
**Görev:** Haber metinlerini 10 ana kategoriye sınıflandırma  

| Kategori | Açıklama |
|----------|----------|
| Gündem | Türkiye, siyaset, seçim |
| Dünya | Uluslararası haberler |
| Spor | Futbol, basketbol, tüm sporlar |
| Ekonomi | Finans, borsa, enerji |
| Teknoloji | Bilim & teknoloji |
| Sağlık | Sağlık haberleri |
| Kültür-Sanat | Müzik, sinema, magazin |
| Eğitim | Eğitim haberleri |
| Yaşam | Seyahat, moda, astroloji |
| Otomobil | Otomobil & savunma |

## 1. Kurulum ve Import

In [25]:
import sys
import os

# ── Google Colab tespiti ──────────────────────────────────────────────
IS_COLAB = "google.colab" in sys.modules

# ── Colab'da eksik paketleri kur ─────────────────────────────────────
if IS_COLAB:
    os.system("pip install -q transformers datasets scikit-learn tqdm matplotlib")
    print("✓ Paketler kuruldu")

import csv
import json
import time
import random
import numpy as np
from pathlib import Path
from collections import Counter
from IPython.display import display, HTML, clear_output

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Ortam   : {'Google Colab' if IS_COLAB else 'Yerel'}")


PyTorch : 2.10.0+cpu
CUDA    : False
Ortam   : Yerel


## 2. Ayarlar

In [15]:
# ── Dizin ayarları ───────────────────────────────────────────────────
if IS_COLAB:
    BASE_DIR = Path("/content/news_project")
    BASE_DIR.mkdir(exist_ok=True)
    # İsteğe bağlı: modeli Drive'a kaydetmek için aşağıdaki iki satırı aç
    # from google.colab import drive; drive.mount("/content/drive")
    # MODEL_DIR = Path("/content/drive/MyDrive/distilbert_turkish_news")
else:
    BASE_DIR = Path(r"c:\Users\LENOVO\Desktop\Aktif Projeler\news")

DATA_DIR  = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Colab'da veri setini HuggingFace'ten indir ───────────────────────
if IS_COLAB and not (DATA_DIR / "train.csv").exists():
    from datasets import load_dataset
    print("📥 TR-News veri seti indiriliyor...")
    ds = load_dataset("batubayk/TR-News")
    for split in ["train", "validation", "test"]:
        ds[split].to_pandas().to_csv(DATA_DIR / f"{split}.csv", index=False)
        print(f"  ✓ {split}.csv ({len(ds[split]):,} satır)")
elif IS_COLAB:
    print("✓ Veri seti zaten mevcut, atlanıyor")

# ── Device ve GPU ayarları ───────────────────────────────────────────
# RTX 4070 (8GB VRAM) için optimize edilmiş
cuda_available = torch.cuda.is_available()

if cuda_available:
    # GPU cihazını seç (0 = ilk GPU, genellikle RTX 4070)
    device = torch.device("cuda:0")
    torch.cuda.set_device(0)
    
    # GPU memory optimizasyonu
    torch.cuda.empty_cache()
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True  # cuDNN auto-tuner
    torch.backends.cudnn.deterministic = False  # daha hızlı ama az farklı sonuçlar
else:
    device = torch.device("cpu")

# ── Model ve eğitim ayarları ─────────────────────────────────────────
MODEL_NAME   = "dbmdz/distilbert-base-turkish-cased"
MAX_LEN      = 256

# RTX 4070 (8GB VRAM) için optimize: 32-48 batch size makul
# Mixed precision training ile daha da yüksek olabilir
if cuda_available:
    BATCH_SIZE = 32  # RTX 4070 8GB için optimal
else:
    BATCH_SIZE = 8

EPOCHS       = 3
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
SEED         = 42
GRADIENT_ACCUMULATION_STEPS = 1  # 4070 için genellikle 1 yeterli

csv.field_size_limit(10**7)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Print sistem bilgisi ────────────────────────────────────────────
print("=" * 70)
print("🖥️  SİSTEM BİLGİSİ")
print("=" * 70)
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA Avail   : {cuda_available}")

if cuda_available:
    print(f"GPU Device   : {torch.cuda.get_device_name(0)}")
    print(f"GPU Index    : cuda:0")
    
    # GPU Memory bilgisi
    gpu_props = torch.cuda.get_device_properties(0)
    total_memory = gpu_props.total_memory / 1024**3
    print(f"GPU Memory   : {total_memory:.2f} GB")
    
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    print(f"Memory Used  : {allocated:.2f} GB / {reserved:.2f} GB reserved")
else:
    print(f"GPU Device   : None (using CPU)")

print(f"\nEğitim Ayarları:")
print(f"  • Batch Size : {BATCH_SIZE}")
print(f"  • Epochs     : {EPOCHS}")
print(f"  • LR         : {LR}")
print(f"  • Warmup     : {WARMUP_STEPS}")
print(f"  • Grad Accum : {GRADIENT_ACCUMULATION_STEPS}")
print(f"  • Device     : {device}")
print("=" * 70)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}  (var: {DATA_DIR.exists()})")
print(f"MODEL_DIR  : {MODEL_DIR}  (var: {MODEL_DIR.exists()})")

🖥️  SİSTEM BİLGİSİ
PyTorch      : 2.10.0+cpu
CUDA Avail   : False
GPU Device   : None (using CPU)

Eğitim Ayarları:
  • Batch Size : 8
  • Epochs     : 3
  • LR         : 2e-05
  • Warmup     : 500
  • Grad Accum : 1
  • Device     : cpu
BASE_DIR   : c:\Users\LENOVO\Desktop\Aktif Projeler\news
DATA_DIR   : c:\Users\LENOVO\Desktop\Aktif Projeler\news\data  (var: True)
MODEL_DIR  : c:\Users\LENOVO\Desktop\Aktif Projeler\news\model  (var: True)


In [24]:
# ═════════════════════════════════════════════════════════════════════════════
# 🔍 DETAYLI GPU ANALİZİ
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 75)
print("🔬 DETAYLI GPU ANALİZİ VE AYARLAR")
print("=" * 75)

# ─────────────────────────────────────────────────────────────────────────────
# 1. CUDA AVAILABILITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
print("\n1️⃣  CUDA AVAILABILITY CHECK")
print("-" * 75)

cuda_avail = torch.cuda.is_available()
cuda_version = torch.version.cuda
cudnn_version = torch.backends.cudnn.version()

print(f"   ✓ CUDA İndir Olabilir        : {cuda_avail}")
print(f"   ✓ CUDA Versiyonu             : {cuda_version}")
print(f"   ✓ CuDNN Versiyonu            : {cudnn_version}")
print(f"   ✓ cuDNN Enabled              : {torch.backends.cudnn.enabled}")
print(f"   ✓ cuDNN Benchmark Mode       : {torch.backends.cudnn.benchmark}")
print(f"   ✓ cuDNN Deterministic        : {torch.backends.cudnn.deterministic}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. GPU DEVICES
# ─────────────────────────────────────────────────────────────────────────────
print("\n2️⃣  GPU CİHAZLARI")
print("-" * 75)

if cuda_avail:
    num_gpus = torch.cuda.device_count()
    print(f"   ✓ GPU Sayısı                 : {num_gpus}")
    
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"   ✓ GPU {i}                     : {gpu_name}")
        
        # Compute capability
        capability = torch.cuda.get_device_capability(i)
        print(f"     └─ Compute Capability      : {capability[0]}.{capability[1]} (CC)")
else:
    print(f"   ❌ GPU Bulunamadı - CPU kullanılacak")

# ─────────────────────────────────────────────────────────────────────────────
# 3. ACTIVE DEVICE DETAILS
# ─────────────────────────────────────────────────────────────────────────────
print("\n3️⃣  AKTIF GPU CİHAZI")
print("-" * 75)

if cuda_avail:
    current_device = torch.cuda.current_device()
    print(f"   ✓ Mevcut Device Index        : {current_device}")
    print(f"   ✓ Device (Torch)             : {device}")
    
    gpu_props = torch.cuda.get_device_properties(current_device)
    
    print(f"\n   🖥️  GPU ÖZELLİKLERİ:")
    print(f"      • GPU Name               : {gpu_props.name}")
    print(f"      • Compute Capability     : {gpu_props.major}.{gpu_props.minor}")
    print(f"      • Multiprocessor Count   : {gpu_props.multi_processor_count}")
    print(f"      • Max Threads per Block  : {gpu_props.max_threads_per_block}")
    print(f"      • Max Block Dims         : {gpu_props.max_block_dims}")
    print(f"      • Total Memory           : {gpu_props.total_memory / 1024**3:.2f} GB")
else:
    print(f"   ❌ GPU seçilemedi")

# ─────────────────────────────────────────────────────────────────────────────
# 4. MEMORY ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
print("\n4️⃣  HAFIZA ANALİZİ")
print("-" * 75)

if cuda_avail:
    # Memory properties
    total_memory = torch.cuda.get_device_properties(0).total_memory
    
    # Current memory usage
    allocated = torch.cuda.memory_allocated(0)
    reserved = torch.cuda.memory_reserved(0)
    free = total_memory - reserved
    
    total_gb = total_memory / 1024**3
    allocated_gb = allocated / 1024**3
    reserved_gb = reserved / 1024**3
    free_gb = free / 1024**3
    
    print(f"   ✓ Toplam Bellek              : {total_gb:.2f} GB")
    print(f"   ✓ Ayrılan Bellek             : {allocated_gb:.2f} GB ({allocated_gb/total_gb*100:.1f}%)")
    print(f"   ✓ Rezerve Bellek             : {reserved_gb:.2f} GB ({reserved_gb/total_gb*100:.1f}%)")
    print(f"   ✓ Boş Bellek                 : {free_gb:.2f} GB ({free_gb/total_gb*100:.1f}%)")
    
    # Memory efficiency
    if reserved > 0:
        efficiency = allocated / reserved * 100
        print(f"\n   📊 Bellek Verimliliği         : {efficiency:.1f}%")
        if efficiency < 50:
            print(f"      ⚠️  Uyarı: Bellek fragmentation yüksek")
else:
    print(f"   ❌ GPU hafızası ölçülemedi")

# ─────────────────────────────────────────────────────────────────────────────
# 5. OPTIMIZATION SETTINGS
# ─────────────────────────────────────────────────────────────────────────────
print("\n5️⃣  OPTİMİZASYON AYARLARI")
print("-" * 75)

print(f"   ✓ PyTorch Version            : {torch.__version__}")
print(f"   ✓ Device Type                : {'GPU (CUDA)' if cuda_avail else 'CPU'}")
print(f"   ✓ CuDNN Benchmark Enabled    : {torch.backends.cudnn.benchmark}")
print(f"   ✓ CuDNN Deterministic        : {torch.backends.cudnn.deterministic}")

if cuda_avail and BATCH_SIZE == 32:
    print(f"   ✓ Batch Size                 : {BATCH_SIZE} (RTX 4070 8GB için optimal)")
elif cuda_avail:
    print(f"   ✓ Batch Size                 : {BATCH_SIZE}")
else:
    print(f"   ✓ Batch Size                 : {BATCH_SIZE} (CPU)")

print(f"   ✓ Mixed Precision Support    : {torch.cuda.is_available() and hasattr(torch.cuda, 'amp')}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. TRAINING OPTIMIZATION CHECK
# ─────────────────────────────────────────────────────────────────────────────
print("\n6️⃣  EĞİTİM OPTİMİZASYONU")
print("-" * 75)

print(f"   ✓ Model Device              : {next(model.parameters()).device}")
print(f"   ✓ Model Parameters          : {sum(p.numel() for p in model.parameters()):,}")
print(f"   ✓ Trainable Parameters      : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   ✓ Learning Rate             : {LR}")
print(f"   ✓ Warmup Steps              : {WARMUP_STEPS}")
print(f"   ✓ Gradient Accumulation     : {GRADIENT_ACCUMULATION_STEPS}")

if cuda_avail and BATCH_SIZE >= 32:
    print(f"\n   ✅ Bu Konfigürasyon RTX 4070 için optimize edilmiştir!")
    print(f"      • Mixed precision training: Evet")
    print(f"      • Gradient clipping: Evet")
    print(f"      • cuDNN auto-tuning: Evet")
else:
    print(f"\n   ⚠️  CPU Training: Daha yavaş olacak")

# ─────────────────────────────────────────────────────────────────────────────
# 7. PERFORMANCE ESTIMATE
# ─────────────────────────────────────────────────────────────────────────────
print("\n7️⃣  PERFORMANS TAHMİNİ")
print("-" * 75)

total_samples = len(train_loader) * BATCH_SIZE * EPOCHS
steps_per_epoch = len(train_loader)
estimated_time_per_epoch = steps_per_epoch * 0.5  # ~0.5 saniye per step estimate

print(f"   ✓ Toplam Eğitim Örneği      : {total_samples:,}")
print(f"   ✓ Adım/Epoch                : {steps_per_epoch:,}")
print(f"   ✓ Tahmini Süre/Epoch        : ~{estimated_time_per_epoch/60:.1f} dakika")
print(f"   ✓ Tahmini Toplam Süre       : ~{estimated_time_per_epoch * EPOCHS / 3600:.1f} saat")

if cuda_avail:
    print(f"\n   ✅ GPU ile hızlı eğitim bekleniyor!")
else:
    print(f"\n   ⚠️  CPU ile eğitim GEÇ olacak - GPU tavsiye edilir")

print("\n" + "=" * 75)



🔬 DETAYLI GPU ANALİZİ VE AYARLAR

1️⃣  CUDA AVAILABILITY CHECK
---------------------------------------------------------------------------
   ✓ CUDA İndir Olabilir        : False
   ✓ CUDA Versiyonu             : None
   ✓ CuDNN Versiyonu            : None
   ✓ cuDNN Enabled              : True
   ✓ cuDNN Benchmark Mode       : False
   ✓ cuDNN Deterministic        : False

2️⃣  GPU CİHAZLARI
---------------------------------------------------------------------------
   ❌ GPU Bulunamadı - CPU kullanılacak

3️⃣  AKTIF GPU CİHAZI
---------------------------------------------------------------------------
   ❌ GPU seçilemedi

4️⃣  HAFIZA ANALİZİ
---------------------------------------------------------------------------
   ❌ GPU hafızası ölçülemedi

5️⃣  OPTİMİZASYON AYARLARI
---------------------------------------------------------------------------
   ✓ PyTorch Version            : 2.10.0+cpu
   ✓ Device Type                : CPU
   ✓ CuDNN Benchmark Enabled    : False
   ✓ CuDNN Deter

## 3. Kategori Eşleme (122 → 10)

In [16]:
CATEGORY_MAP = {
    # Gündem / Siyaset
    "Türkiye": "Gündem", "turkiye": "Gündem", "Gündem": "Gündem",
    "siyaset": "Gündem", "Polemik": "Gündem",
    "2019 Yerel Seçim": "Gündem", "23 Haziran seçimleri": "Gündem",
    "secim_2015": "Gündem", "secim": "Gündem", "Seçim": "Gündem",
    "Seçim 2015": "Gündem", "yerel_yonetimler": "Gündem",
    "H. Bunu Konuşuyor": "Gündem", "15 Temmuz davaları": "Gündem",
    "Haberin Var Mı?": "Gündem", "Kadına Şiddet": "Gündem",
    "sokak": "Gündem", "Ortak Gelecek": "Gündem",

    # Dünya
    "Dünya": "Dünya", "dunya": "Dünya", "Dünyadan": "Dünya",
    "BBC": "Dünya", "english": "Dünya",

    # Spor
    "Spor": "Spor", "spor": "Spor",
    "futbol": "Spor", "Futbol": "Spor",
    "basketbol": "Spor", "Basketbol": "Spor",
    "Voleybol": "Spor", "Tenis": "Spor", "Hentbol": "Spor",
    "Motor Sporları": "Spor", "Olimpiyat": "Spor",
    "diger_sporlar": "Spor", "İddaa": "Spor",
    "EURO 2016": "Spor", "2016 Avrupa Şampiyonası": "Spor",
    "2018 Dünya Kupası": "Spor", "Dünya Kupası 2018": "Spor",
    "2016 Rio Olimpiyatları": "Spor", "foto_spor": "Spor",
    "2018 DK Tarihçe": "Spor",

    # Ekonomi
    "Ekonomi": "Ekonomi", "ekonomi": "Ekonomi",
    "Makro Ekonomi": "Ekonomi", "Para": "Ekonomi",
    "Emlak": "Ekonomi", "Enerji": "Ekonomi",
    "Döviz": "Ekonomi", "Borsa": "Ekonomi", "Altın": "Ekonomi",
    "Sigorta": "Ekonomi", "Sosyal Güvenlik": "Ekonomi",
    "Bitcoin": "Ekonomi", "Girişimcilik": "Ekonomi",
    "Akdeniz Ekonomi Forumu": "Ekonomi",
    "Ege Ekonomik Forum 2017": "Ekonomi",
    "AIRPORT": "Ekonomi",

    # Teknoloji & Bilim
    "Teknoloji": "Teknoloji", "bilim_ve_teknoloji": "Teknoloji",
    "uzay": "Teknoloji",

    # Sağlık
    "Sağlık": "Sağlık", "saglik": "Sağlık",

    # Kültür & Sanat & Magazin
    "Sanat": "Kültür-Sanat", "Kültür-Sanat": "Kültür-Sanat",
    "kultur-sanat": "Kültür-Sanat", "foto_kultur_sanat": "Kültür-Sanat",
    "Müzik": "Kültür-Sanat", "konser": "Kültür-Sanat",
    "Televizyon": "Kültür-Sanat", "sinema": "Kültür-Sanat",
    "kitap": "Kültür-Sanat", "Biyografi": "Kültür-Sanat",
    "Fiskos": "Kültür-Sanat", "Magazin": "Kültür-Sanat",
    "Cemiyet Hayatı": "Kültür-Sanat", "Medya": "Kültür-Sanat",
    "Röportajlar": "Kültür-Sanat", "Özel Röportajlar": "Kültür-Sanat",
    "Özel": "Kültür-Sanat", "etkinlik": "Kültür-Sanat",

    # Eğitim
    "Eğitim": "Eğitim", "egitim": "Eğitim",

    # Yaşam
    "Yaşam": "Yaşam", "yasam": "Yaşam", "İş-Yaşam": "Yaşam",
    "calisma_yasami": "Yaşam", "Seyahat": "Yaşam", "gezi": "Yaşam",
    "Turizm": "Yaşam", "cevre": "Yaşam",
    "surdurulebilir_yasam": "Yaşam", "Ramazan": "Yaşam",
    "Tarifler": "Yaşam", "Yemek Yapma": "Yaşam",
    "Nasıl Yapılır": "Yaşam", "Alışveriş": "Yaşam",
    "Güvenli Alışveriş": "Yaşam", "Moda": "Yaşam", "moda": "Yaşam",
    "Evde Ek İş": "Yaşam", "Sıfır Atık": "Yaşam",
    "Merak Edilenler": "Yaşam", "Nedir": "Yaşam",
    "Astroloji": "Yaşam", "astroloji": "Yaşam",
    "Şipşak": "Yaşam", "Takılın": "Yaşam",
    "MOTY 2019": "Yaşam",

    # Otomobil
    "Otomobil": "Otomobil", "otomobil": "Otomobil",
    "Savunma Sanayi": "Otomobil",
}

SKIP_TOPICS = {"", "diger", "Diğer", "General", "Yazı Dizisi",
               "yazi_dizileri", "pazar_yazilari",
               "cumhuriyet_ege", "cumhuriyet_pazar"}

print(f"Eşleme tablosundaki ham etiket sayısı: {len(CATEGORY_MAP)}")
print(f"Ana kategori sayısı: {len(set(CATEGORY_MAP.values()))}")

Eşleme tablosundaki ham etiket sayısı: 113
Ana kategori sayısı: 10


## 4. Veri Yükleme ve Analiz

In [17]:
def load_csv(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            topic = row.get("topic", "").strip()
            if topic in SKIP_TOPICS:
                continue
            category = CATEGORY_MAP.get(topic)
            if category is None:
                continue
            title = (row.get("title") or "").strip()
            abstract = (row.get("abstract") or "").strip()
            text = f"{title}. {abstract}" if abstract else title
            if len(text) < 10:
                continue
            rows.append({"text": text, "category": category})
    return rows

train_rows = load_csv(DATA_DIR / "train.csv")
val_rows = load_csv(DATA_DIR / "validation.csv")
test_rows = load_csv(DATA_DIR / "test.csv")

print(f"Train: {len(train_rows):,}")
print(f"Val:   {len(val_rows):,}")
print(f"Test:  {len(test_rows):,}")
print(f"Toplam: {len(train_rows)+len(val_rows)+len(test_rows):,}")

Train: 274,106
Val:   14,415
Test:  15,192
Toplam: 303,713


In [18]:
# Kategori dağılımı
categories = sorted(set(r["category"] for r in train_rows))
cat2id = {c: i for i, c in enumerate(categories)}
id2cat = {i: c for c, i in cat2id.items()}
num_labels = len(categories)

print(f"\nKategori sayısı: {num_labels}")
print(f"{'Kategori':<15} {'Train':>7} {'Val':>7} {'Test':>7}")
print("-" * 40)

train_dist = Counter(r["category"] for r in train_rows)
val_dist = Counter(r["category"] for r in val_rows)
test_dist = Counter(r["category"] for r in test_rows)

for c in categories:
    print(f"{c:<15} {train_dist[c]:>7,} {val_dist[c]:>7,} {test_dist[c]:>7,}")

# Label mapping kaydet
label_map = {"cat2id": cat2id, "id2cat": {str(k): v for k, v in id2cat.items()}}
with open(MODEL_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print(f"\n✓ label_map.json kaydedildi")


Kategori sayısı: 10
Kategori          Train     Val    Test
----------------------------------------
Dünya            40,952   2,099   2,294
Ekonomi          20,433   1,123   1,142
Eğitim            5,082     290     303
Gündem          122,827   6,529   6,781
Kültür-Sanat     11,536     580     590
Otomobil          1,513      94      95
Sağlık           16,175     872     928
Spor             30,871   1,549   1,701
Teknoloji         7,011     364     376
Yaşam            17,706     915     982

✓ label_map.json kaydedildi


In [19]:
# Örnek veriler
print("\n--- Örnek Veriler ---")
for i in range(3):
    r = train_rows[i]
    print(f"\n[{i+1}] Kategori: {r['category']}")
    print(f"    Metin: {r['text'][:120]}...")


--- Örnek Veriler ---

[1] Kategori: Kültür-Sanat
    Metin: Tuğba Özerk'ten Deniz Akkaya’ya 100 bin TL’lik dava. Şarkıcı Tuğba Özerk, annesine canlı yayında yönelttiği soru için De...

[2] Kategori: Gündem
    Metin: Bahçeli: Cepheleşme keskinleşirse MHP buna tepkisiz kalmayacaktır. MHP Lideri Bahçeli, cumhurbaşkanlığı hükümet sistemin...

[3] Kategori: Dünya
    Metin: YPG ve ABD omuz omuza. Pentagon, ABD askerlerinin Suriye’de Kürt birliklere eşlik etmesini önerdi....


## 5. Dataset ve DataLoader

In [20]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = [r["text"] for r in train_rows]
train_labels = [cat2id[r["category"]] for r in train_rows]
val_texts = [r["text"] for r in val_rows]
val_labels = [cat2id[r["category"]] for r in val_rows]
test_texts = [r["text"] for r in test_rows]
test_labels = [cat2id[r["category"]] for r in test_rows]

train_ds = NewsDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds = NewsDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_ds = NewsDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, num_workers=0)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches:   {len(val_loader)}")
print(f"✓ Test batches:  {len(test_loader)}")

# Örnek tokenize
sample = train_ds[0]
print(f"\nÖrnek input_ids shape: {sample['input_ids'].shape}")
print(f"Örnek attention_mask shape: {sample['attention_mask'].shape}")
print(f"Örnek label: {sample['labels'].item()} → {id2cat[sample['labels'].item()]}")

Tokenizer yükleniyor...


2026-03-12 21:57:44,506 - INFO - HTTP Request: HEAD https://huggingface.co/dbmdz/distilbert-base-turkish-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-12 21:57:44,517 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dbmdz/distilbert-base-turkish-cased/8ecd4d034c2612d4c5940795b4f2552a9f3543d6/config.json "HTTP/1.1 200 OK"
2026-03-12 21:57:44,756 - INFO - HTTP Request: HEAD https://huggingface.co/dbmdz/distilbert-base-turkish-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-12 21:57:44,757 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-12 21:57:44,765 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dbmdz/distilbert-base-turkish-cased/8ecd4d034c2612d4c5940795b4f2552a9f3543d6/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-12 21:57:45,222 - INFO - HTTP Request: GET h

✓ Train batches: 34264
✓ Val batches:   901
✓ Test batches:  950

Örnek input_ids shape: torch.Size([256])
Örnek attention_mask shape: torch.Size([256])
Örnek label: 4 → Kültür-Sanat


## 6. Model Yükleme

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)
model.to(device)

# Parametre sayısı
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Toplam parametre: {total_params:,}")
print(f"Eğitilebilir:     {trainable_params:,}")
print(f"Model hazır ({device})")

2026-03-12 21:57:45,989 - INFO - HTTP Request: HEAD https://huggingface.co/dbmdz/distilbert-base-turkish-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-12 21:57:45,998 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dbmdz/distilbert-base-turkish-cased/8ecd4d034c2612d4c5940795b4f2552a9f3543d6/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6143.43it/s]
DistilBertForSequenceClassification LOAD REPORT from: dbmdz/distilbert-base-turkish-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:ca

Toplam parametre: 68,096,266
Eğitilebilir:     68,096,266
Model hazır (cpu)


## 7. Eğitim Fonksiyonları

In [22]:
def evaluate(model, dataloader, device, desc='Evaluating'):
    model.eval()
    preds_all, labels_all = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc, leave=False, bar_format='{l_bar}{bar:20}| {n_fmt}/{total_fmt}'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=-1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    return np.array(preds_all), np.array(labels_all), avg_loss

print('\u2713 Fonksiyonlar tanımlandı')


✓ Fonksiyonlar tanımlandı



# Model yüklenmemişse otomatik yükle
if 'model' not in vars() or model is None:
    print('Model henüz yüklenmemiş, yükleniyor...')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model.to(device)
    print(f'\u2713 Model yüklendi ({device})')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, total_steps)

print(f'Toplam adım: {total_steps:,}')
print(f'Warmup: {WARMUP_STEPS} | Epoch: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}')
print()

best_f1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
global_step = 0
t_start = time.time()

# === ANA PROGRESS BAR (toplam %) ===
main_bar = tqdm(total=total_steps, desc='\U0001f4ca Toplam Eğitim', unit='step',
                bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    # Epoch progress bar
    epoch_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=True,
                    bar_format='{l_bar}{bar:25}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

    for step, batch in enumerate(epoch_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        epoch_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        global_step += 1
        main_bar.update(1)

        avg_loss = epoch_loss / (step + 1)
        pct = global_step / total_steps * 100
        elapsed = time.time() - t_start
        eta = elapsed / global_step * (total_steps - global_step) if global_step > 0 else 0
        eta_min = int(eta // 60)
        eta_sec = int(eta % 60)

        epoch_bar.set_postfix({'loss': f'{avg_loss:.4f}', '%': f'{pct:.1f}'})
        main_bar.set_postfix({'loss': f'{avg_loss:.4f}', 'ETA': f'{eta_min}m{eta_sec:02d}s'})

    epoch_bar.close()
    avg_train_loss = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)

    # Validation
    val_bar = tqdm(val_loader, desc=f'  Val {epoch+1}/{EPOCHS}', leave=False,
                  bar_format='{l_bar}{bar:20}| {n_fmt}/{total_fmt}')
    model.eval()
    preds_all, labels_all, total_val_loss = [], [], 0
    with torch.no_grad():
        for batch in val_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            lbl = batch['labels'].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=lbl)
            total_val_loss += out.loss.item()
            preds_all.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
            labels_all.extend(lbl.cpu().numpy())
    val_bar.close()

    val_loss = total_val_loss / len(val_loader)
    preds_np = np.array(preds_all)
    labels_np = np.array(labels_all)
    acc = accuracy_score(labels_np, preds_np)
    f1 = f1_score(labels_np, preds_np, average='macro')
    history['val_loss'].append(val_loss)
    history['val_acc'].append(acc)
    history['val_f1'].append(f1)

    elapsed_total = time.time() - t_start
    em, es = int(elapsed_total // 60), int(elapsed_total % 60)

    print(f'\n{"="*60}')
    print(f'Epoch {epoch+1}/{EPOCHS}  |  Süre: {em}m{es:02d}s  |  %{pct:.1f}')
    print(f'  Train Loss: {avg_train_loss:.4f}')
    print(f'  Val Loss:   {val_loss:.4f}')
    print(f'  Val Acc:    {acc:.4f}')
    print(f'  Val F1:     {f1:.4f}')

    # ── Her epoch sonunda checkpoint kaydet ──────────────────────────
    ckpt_dir = MODEL_DIR / f"checkpoint-epoch-{epoch+1}"
    ckpt_dir.mkdir(exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    torch.save({
        'epoch': epoch + 1,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'global_step': global_step,
        'best_f1': best_f1,
        'history': history,
        'val_f1': f1,
        'val_acc': acc,
    }, ckpt_dir / 'training_state.pt')
    print(f'  \U0001f4be Checkpoint kaydedildi: checkpoint-epoch-{epoch+1}/')

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print(f'  \u2713 En iyi model kaydedildi! (F1: {f1:.4f})')
    print(f'{"="*60}\n')

main_bar.close()

total_time = time.time() - t_start
tm, ts = int(total_time // 60), int(total_time % 60)
print(f'\n\u2705 Eğitim tamamlandı! Toplam süre: {tm}m{ts:02d}s')
print(f'\u2705 En iyi F1: {best_f1:.4f}')


In [23]:
# ═════════════════════════════════════════════════════════════════════════════
# 🎯 PRODUCTION-READY ADVANCED TRAINING LOOP
# ═════════════════════════════════════════════════════════════════════════════
# Hatasız, robust, endüstri standardı eğitim sistemi

import logging
import gc
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class TrainingState:
    """Eğitim durumu ve metadata"""
    epoch: int = 0
    global_step: int = 0
    best_f1: float = 0.0
    best_epoch: int = 0
    num_batches_failed: int = 0

class EarlyStoppingCallback:
    """Early stopping mekanizması"""
    def __init__(self, patience: int = 3, metric: str = 'f1', min_delta: float = 1e-4):
        self.patience = patience
        self.metric = metric
        self.patience_counter = 0
        self.best_value = 0.0
        self.min_delta = min_delta
    
    def __call__(self, metrics: Dict[str, float]) -> bool:
        current = metrics.get(self.metric, 0.0)
        if current > self.best_value + self.min_delta:
            self.best_value = current
            self.patience_counter = 0
            return False
        else:
            self.patience_counter += 1
            return self.patience_counter >= self.patience

class BestModelCheckpoint:
    """Best model checkpoint yönetisi"""
    def __init__(self, save_dir: Path, metric: str = 'f1'):
        self.save_dir = Path(save_dir)
        self.metric = metric
        self.best_value = 0.0
        self.save_dir.mkdir(parents=True, exist_ok=True)
    
    def save(self, model, tokenizer, metrics: Dict[str, float], info: Dict) -> bool:
        try:
            current = metrics.get(self.metric, 0.0)
            
            if current > self.best_value:
                self.best_value = current
                
                # Dizin oluştur
                model_dir = self.save_dir / "pytorch_model"
                tokenizer_dir = self.save_dir / "tokenizer"
                model_dir.mkdir(parents=True, exist_ok=True)
                tokenizer_dir.mkdir(parents=True, exist_ok=True)
                
                # Model kaydet
                model.save_pretrained(str(model_dir))
                tokenizer.save_pretrained(str(tokenizer_dir))
                
                # Metadata kaydet
                metadata = {
                    'metrics': metrics,
                    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                    **info
                }
                with open(self.save_dir / "checkpoint_info.json", 'w', encoding='utf-8') as f:
                    json.dump(metadata, f, indent=2, ensure_ascii=False)
                
                logger.info(f"✓ Best model kaydedildi! {self.metric}={current:.4f}")
                return True
        except Exception as e:
            logger.error(f"❌ Checkpoint kaydetme hatası: {e}")
        return False

class MetricsLogger:
    """Metrik kayıt ve analiz"""
    def __init__(self):
        self.history: Dict[str, List[float]] = {
            'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [],
            'val_precision': [], 'val_recall': [], 'learning_rate': []
        }
    
    def log(self, **metrics):
        for key, value in metrics.items():
            if key not in self.history:
                self.history[key] = []
            self.history[key].append(float(value))

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, train_loader, optimizer, scheduler, device, 
                scaler=None, max_grad_norm: float = 1.0) -> Tuple[float, int]:
    """
    Tek bir epoch'u eğit - Production ready
    """
    model.train()
    epoch_loss = 0.0
    total_batches = 0
    failed_batches = 0
    
    print(f"   🔄 Training: ", end='', flush=True)
    
    for step, batch in enumerate(train_loader):
        try:
            # Data to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            if scaler is not None:  # Mixed Precision
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )
                    loss = outputs.loss
                
                # Check for NaN/Inf
                if not torch.isfinite(loss):
                    logger.warning(f"⚠️ NaN/Inf loss at step {step}, skipping batch")
                    failed_batches += 1
                    continue
                
                # Backward
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                
            else:  # Standard training
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss
                
                # Check for NaN/Inf
                if not torch.isfinite(loss):
                    logger.warning(f"⚠️ NaN/Inf loss at step {step}, skipping batch")
                    failed_batches += 1
                    continue
                
                # Backward
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                optimizer.step()
            
            # Step scheduler
            scheduler.step()
            optimizer.zero_grad()
            
            epoch_loss += loss.item()
            total_batches += 1
            
            # Progress update
            progress = (step + 1) / len(train_loader)
            if (step + 1) % max(1, len(train_loader) // 5) == 0:
                print(f"{progress * 100:.0f}%→", end='', flush=True)
            
        except RuntimeError as e:
            error_msg = str(e).lower()
            if 'out of memory' in error_msg or 'cuda' in error_msg:
                logger.warning(f"⚠️ GPU memory issue at step {step}, clearing cache...")
                torch.cuda.empty_cache()
                gc.collect()
                failed_batches += 1
            else:
                logger.error(f"❌ RuntimeError step {step}: {e}")
                failed_batches += 1
        
        except Exception as e:
            logger.error(f"❌ Unexpected error at step {step}: {e}")
            failed_batches += 1
    
    print(" ✓")
    
    if failed_batches > 0:
        logger.warning(f"⚠️ {failed_batches} batch(es) skipped due to errors")
    
    avg_loss = epoch_loss / total_batches if total_batches > 0 else 0.0
    return avg_loss, failed_batches

# ─────────────────────────────────────────────────────────────────────────────

def validate(model, val_loader, device) -> Dict[str, float]:
    """
    Validation - Production ready
    """
    model.eval()
    val_loss = 0.0
    preds_list = []
    labels_list = []
    val_batches = 0
    
    print(f"   ✓ Validating: ", end='', flush=True)
    
    try:
        with torch.no_grad():
            for batch in val_loader:
                try:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)
                    
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )
                    
                    # Check for NaN/Inf
                    if not torch.isfinite(outputs.loss):
                        logger.warning(f"⚠️ NaN/Inf loss in validation, skipping batch")
                        continue
                    
                    val_loss += outputs.loss.item()
                    
                    preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
                    preds_list.extend(preds)
                    labels_list.extend(labels.cpu().numpy())
                    val_batches += 1
                    
                except Exception as e:
                    logger.warning(f"⚠️ Validation batch error: {e}")
                    continue
        
        # Metrikleri hesapla
        from sklearn.metrics import precision_score, recall_score
        
        if len(preds_list) == 0 or len(labels_list) == 0:
            logger.warning("⚠️ No valid predictions in validation")
            metrics = {
                'val_loss': 0.0,
                'val_acc': 0.0,
                'val_f1': 0.0,
                'val_precision': 0.0,
                'val_recall': 0.0,
            }
        else:
            preds_np = np.array(preds_list, dtype=np.int32)
            labels_np = np.array(labels_list, dtype=np.int32)
            
            metrics = {
                'val_loss': val_loss / val_batches if val_batches > 0 else 0.0,
                'val_acc': float(accuracy_score(labels_np, preds_np)),
                'val_f1': float(f1_score(labels_np, preds_np, average='macro', zero_division=0)),
                'val_precision': float(precision_score(labels_np, preds_np, average='macro', zero_division=0)),
                'val_recall': float(recall_score(labels_np, preds_np, average='macro', zero_division=0)),
            }
        
        print("✓")
        return metrics
        
    except Exception as e:
        logger.error(f"❌ Validation error: {e}")
        return {
            'val_loss': 0.0,
            'val_acc': 0.0,
            'val_f1': 0.0,
            'val_precision': 0.0,
            'val_recall': 0.0,
        }

# ═════════════════════════════════════════════════════════════════════════════
# MAIN TRAINING LOOP
# ═════════════════════════════════════════════════════════════════════════════

try:
    logger.info("🚀 Training sistemini başlatıyor...")
    
    # ── Pre-flight checks ───────────────────────────────────────────────────
    assert model is not None, "❌ Model yüklenmedi!"
    assert train_loader is not None, "❌ Train loader bulunamadı!"
    assert val_loader is not None, "❌ Validation loader bulunamadı!"
    assert id2cat is not None, "❌ id2cat mapping bulunamadı!"
    assert device is not None, "❌ Device tanımlanmadı!"
    
    logger.info("✓ Ön kontroller passed")
    
    # ── Model setup ─────────────────────────────────────────────────────────
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    model.to(device)
    
    # ── Optimizer ───────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )
    
    # ── Scheduler ───────────────────────────────────────────────────────────
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps
    )
    
    # ── Callbacks ───────────────────────────────────────────────────────────
    early_stop = EarlyStoppingCallback(patience=3, metric='f1', min_delta=1e-4)
    checkpoint = BestModelCheckpoint(MODEL_DIR, metric='f1')
    metrics_logger = MetricsLogger()
    
    # ── Training state ──────────────────────────────────────────────────────
    train_state = TrainingState()
    best_f1 = 0.0
    t_start = time.time()
    
    # ── Print training info ────────────────────────────────────────────────
    logger.info(f"📊 Eğitim Başlıyor")
    logger.info(f"  • Epochs: {EPOCHS}")
    logger.info(f"  • Batch Size: {BATCH_SIZE}")
    logger.info(f"  • Total Steps: {total_steps:,}")
    logger.info(f"  • Learning Rate: {LR}")
    logger.info(f"  • Warmup Steps: {WARMUP_STEPS}")
    logger.info(f"  • Device: {device}")
    logger.info(f"  • Mixed Precision: {use_amp}")
    logger.info(f"  • Model Dir: {MODEL_DIR}")
    print()
    
    # ── EPOCH LOOP ──────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):
        train_state.epoch = epoch + 1
        
        elapsed_so_far = time.time() - t_start
        em_so_far = int(elapsed_so_far // 60)
        es_so_far = int(elapsed_so_far % 60)
        
        print(f"┌─ Epoch {epoch + 1}/{EPOCHS} (Elapsed: {em_so_far}m{es_so_far:02d}s)")
        
        # Training
        try:
            train_loss, failed_batches = train_epoch(
                model, train_loader, optimizer, scheduler, device, scaler
            )
            train_state.num_batches_failed = failed_batches
        except Exception as e:
            logger.error(f"❌ Training failed at epoch {epoch + 1}: {e}")
            raise
        
        # Validation
        try:
            val_metrics = validate(model, val_loader, device)
        except Exception as e:
            logger.error(f"❌ Validation failed at epoch {epoch + 1}: {e}")
            val_metrics = {
                'val_loss': 0.0,
                'val_acc': 0.0,
                'val_f1': 0.0,
                'val_precision': 0.0,
                'val_recall': 0.0,
            }
        
        # Log metrics
        try:
            metrics_logger.log(
                train_loss=train_loss,
                learning_rate=optimizer.param_groups[0]['lr'],
                **val_metrics
            )
        except Exception as e:
            logger.warning(f"⚠️ Metrics logging error: {e}")
        
        # Print results
        elapsed = time.time() - t_start
        em = int(elapsed // 60)
        es = int(elapsed % 60)
        
        print(
            f"├─ Loss: {train_loss:.4f} → {val_metrics['val_loss']:.4f} | "
            f"Acc: {val_metrics['val_acc']:.4f} | "
            f"F1: {val_metrics['val_f1']:.4f} ⭐ | "
            f"Time: {em}m{es:02d}s"
        )
        
        # Save best model
        try:
            saved = checkpoint.save(model, tokenizer, val_metrics, {'epoch': epoch + 1})
            if saved:
                best_f1 = val_metrics['val_f1']
                print(f"├─ ✨ Best Model Saved (F1={best_f1:.4f})")
        except Exception as e:
            logger.warning(f"⚠️ Checkpoint save error: {e}")
        
        # Early stopping check
        should_stop = early_stop(val_metrics)
        if should_stop:
            print(f"└─ ⚠️ Early Stopping at Epoch {epoch + 1}")
            break
        
        print()
    
    # ── Training completed ──────────────────────────────────────────────────
    total_time = time.time() - t_start
    tm = int(total_time // 60)
    ts = int(total_time % 60)
    
    print("═" * 75)
    print("✅ EĞITIM BAŞARIYLA TAMAMLANDI!")
    print("═" * 75)
    print(f"📊 Toplam Süre:           {tm}m{ts:02d}s")
    print(f"⭐ En İyi F1-Score:       {best_f1:.4f}")
    print(f"📁 Model Konumu:          {MODEL_DIR}")
    print(f"🔢 Eğitilen Epoch:        {train_state.epoch}/{EPOCHS}")
    print(f"⚠️  Başarısız Batch:       {train_state.num_batches_failed}")
    print("═" * 75)
    
    # Store for later use
    history = metrics_logger.history
    
except KeyboardInterrupt:
    print("\n⚠️ Eğitim kullanıcı tarafından durduruldu")
    total_time = time.time() - t_start
    tm = int(total_time // 60)
    ts = int(total_time % 60)
    print(f"Çalışma süresi: {tm}m{ts:02d}s")
    print(f"Epoch: {train_state.epoch}/{EPOCHS}")

except AssertionError as e:
    print(f"\n❌ ASSERTION ERROR: {e}")
    logger.error(f"Assertion failed: {e}")

except Exception as e:
    print(f"\n❌ CRITICAL ERROR: {e}")
    logger.error(f"Critical error occurred: {e}", exc_info=True)
    raise

2026-03-12 21:57:46,643 - INFO - 🚀 Training sistemini başlatıyor...
2026-03-12 21:57:46,645 - INFO - ✓ Ön kontroller passed
2026-03-12 21:57:46,676 - INFO - 📊 Eğitim Başlıyor
2026-03-12 21:57:46,677 - INFO -   • Epochs: 3
2026-03-12 21:57:46,678 - INFO -   • Batch Size: 8
2026-03-12 21:57:46,680 - INFO -   • Total Steps: 102,792
2026-03-12 21:57:46,680 - INFO -   • Learning Rate: 2e-05
2026-03-12 21:57:46,680 - INFO -   • Warmup Steps: 500
2026-03-12 21:57:46,680 - INFO -   • Device: cpu
2026-03-12 21:57:46,682 - INFO -   • Mixed Precision: False
2026-03-12 21:57:46,682 - INFO -   • Model Dir: c:\Users\LENOVO\Desktop\Aktif Projeler\news\model



┌─ Epoch 1/3 (Elapsed: 0m00s)
   🔄 Training: 
⚠️ Eğitim kullanıcı tarafından durduruldu
Çalışma süresi: 8m28s
Epoch: 1/3


## 9. Eğitim Grafikleri

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs_range = range(1, EPOCHS + 1)
    
    # Loss
    axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train")
    axes[0].plot(epochs_range, history["val_loss"], "r-o", label="Val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs_range, history["val_acc"], "g-o")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylim(0.5, 1.0)
    axes[1].grid(True, alpha=0.3)
    
    # F1
    axes[2].plot(epochs_range, history["val_f1"], "m-o")
    axes[2].set_title("Validation F1-macro")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylim(0.5, 1.0)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / "training_curves.png"), dpi=150)
    plt.show()
    print("✓ Grafik kaydedildi: training_curves.png")
except ImportError:
    print("matplotlib yüklü değil, grafik atlanıyor.")
    for k, v in history.items():
        print(f"  {k}: {v}")

## 10. Test Sonuçları

In [ ]:
# En iyi modeli yükle
best_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)

preds, labels_arr, test_loss = evaluate(best_model, test_loader, device, desc='Test')
target_names = [id2cat[i] for i in range(num_labels)]

print('Test Classification Report')
print('=' * 60)
print(classification_report(labels_arr, preds, target_names=target_names, digits=4))

test_acc = accuracy_score(labels_arr, preds)
test_f1 = f1_score(labels_arr, preds, average='macro')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1-macro: {test_f1:.4f}')
print(f'Test Loss:     {test_loss:.4f}')


## 11. Confusion Matrix

In [ ]:
try:
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    
    cm = confusion_matrix(labels_arr, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, cmap="Blues")
    
    ax.set_xticks(range(num_labels))
    ax.set_yticks(range(num_labels))
    ax.set_xticklabels(target_names, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(target_names, fontsize=9)
    ax.set_xlabel("Tahmin")
    ax.set_ylabel("Gerçek")
    ax.set_title("Confusion Matrix")
    
    for i in range(num_labels):
        for j in range(num_labels):
            text_color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=text_color, fontsize=8)
    
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / "confusion_matrix.png"), dpi=150)
    plt.show()
    print("✓ Confusion matrix kaydedildi: confusion_matrix.png")
except ImportError:
    print("matplotlib yüklü değil, confusion matrix atlanıyor.")

## 12. Hızlı Test — Kendi Cümlelerinle Dene

In [ ]:
def predict(text):
    best_model.eval()
    enc = tokenizer(text, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = best_model(**enc).logits
    probs = torch.softmax(logits, dim=-1).squeeze()
    pred_id = probs.argmax().item()
    return id2cat[pred_id], probs[pred_id].item(), {id2cat[i]: round(probs[i].item(), 4) for i in range(num_labels)}

test_sentences = [
    "Galatasaray derbide Fenerbahçe'yi 3-1 mağlup etti",
    "Dolar kuru 35 TL'yi aştı, Merkez Bankası faiz kararını açıkladı",
    "Yapay zeka alanında yeni bir çip geliştirildi",
    "Cumhurbaşkanı yeni kabineyi açıkladı",
    "İstanbul'da yeni metro hattı açıldı vatandaşlar memnun",
    "Grip salgınında vaka sayıları artıyor, uzmanlar uyarıyor",
    "Cannes Film Festivali'nde Türk yönetmene ödül",
    "YKS başvuruları bugün sona eriyor",
    "BMW yeni elektrikli SUV modelini tanıttı",
    "Rusya-Ukrayna savaşında yeni gelişmeler",
]

print('Tahmin Sonuçları')
print('=' * 70)
for s in test_sentences:
    cat, conf, _ = predict(s)
    print(f'  [{cat:<14}] (%{conf*100:.1f}) {s}')


## 13. Model Bilgisi Kaydet

In [ ]:
# Eğitim bilgilerini kaydet
training_info = {
    "model_name": MODEL_NAME,
    "num_labels": num_labels,
    "categories": categories,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "best_val_f1": best_f1,
    "test_accuracy": test_acc,
    "test_f1_macro": test_f1,
    "train_samples": len(train_rows),
    "val_samples": len(val_rows),
    "test_samples": len(test_rows),
    "history": history,
    "device": str(device),
}

with open(MODEL_DIR / "training_info.json", "w", encoding="utf-8") as f:
    json.dump(training_info, f, ensure_ascii=False, indent=2)

print("✓ Model kaydedildi:", MODEL_DIR)
print("✓ training_info.json kaydedildi")
print("✓ label_map.json kaydedildi")
print(f"\nModel dosyaları:")
for f in sorted(MODEL_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<30} {size_mb:.1f} MB")